# CRN Surrogate – End-to-End Illustration

Demonstrates the full pipeline on small numbers: generate random CRNs, simulate SSA,
train a tiny model, and evaluate on novel topologies.  Runs in under 5 minutes on CPU.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

repo_root = Path().resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from crn_surrogate.configs.model_config import EncoderConfig, ModelConfig, SDEConfig
from crn_surrogate.configs.training_config import SchedulerType, TrainingConfig
from crn_surrogate.data.dataset import CRNTrajectoryDataset, TrajectoryItem
from crn_surrogate.data.generation.mass_action_generator import (
    MassActionCRNGenerator,
    MassActionGeneratorConfig,
)
from crn_surrogate.encoder.bipartite_gnn import BipartiteGNNEncoder
from crn_surrogate.encoder.tensor_repr import crn_to_tensor_repr
from crn_surrogate.simulation.gillespie import GillespieSSA
from crn_surrogate.simulation.trajectory import Trajectory
from crn_surrogate.simulator.neural_sde import CRNNeuralSDE
from crn_surrogate.simulator.sde_solver import EulerMaruyamaSolver
from crn_surrogate.training.trainer import Trainer

torch.manual_seed(0)
DEVICE = torch.device("cpu")  # keep small for illustration
MAX_N_SPECIES = 3
MAX_N_REACTIONS = 6
T_MAX = 10.0
N_TIME_POINTS = 30
time_grid = torch.linspace(0.0, T_MAX, N_TIME_POINTS)

## Section 1: Generate a few random CRNs

In [ ]:
gen_cfg = MassActionGeneratorConfig(
    n_species_range=(1, MAX_N_SPECIES),
    n_reactions_range=(2, MAX_N_REACTIONS),
)
gen = MassActionCRNGenerator(gen_cfg)

demo_crns = gen.sample_batch(5)
for i, crn in enumerate(demo_crns):
    print(f"\nCRN {i}: {crn}")
    for j, rxn in enumerate(crn.reactions):
        print(f"  rxn {j}: stoich={rxn.stoichiometry.tolist()}  prop={rxn.propensity}")

In [ ]:
# Visualise bipartite graphs for 2 CRNs
try:
    import networkx as nx

    def draw_crn_bipartite(crn, ax, title=""):
        G = nx.DiGraph()
        species_nodes = [f"S{s}" for s in range(crn.n_species)]
        reaction_nodes = [f"R{r}" for r in range(crn.n_reactions)]
        G.add_nodes_from(species_nodes, bipartite=0)
        G.add_nodes_from(reaction_nodes, bipartite=1)
        for r, rxn in enumerate(crn.reactions):
            for s in range(crn.n_species):
                net = rxn.stoichiometry[s].item()
                if net < 0:
                    G.add_edge(f"S{s}", f"R{r}", weight=abs(net))
                elif net > 0:
                    G.add_edge(f"R{r}", f"S{s}", weight=net)
        pos = nx.bipartite_layout(G, species_nodes)
        nx.draw(G, pos, ax=ax, with_labels=True, node_color="lightblue",
                node_size=600, arrows=True, font_size=8)
        ax.set_title(title)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for i in range(2):
        draw_crn_bipartite(demo_crns[i], axes[i], title=f"CRN {i}")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("networkx not installed – skipping graph visualisation")

## Section 2: Simulate SSA

In [ ]:
ssa = GillespieSSA()
M_SSA = 10
demo_trajs = []
demo_inits = []
for crn in demo_crns:
    x0 = gen.sample_initial_state(crn)
    demo_inits.append(x0)
    traj_list = ssa.simulate_batch(
        stoichiometry=crn.stoichiometry_matrix,
        propensity_fn=crn.evaluate_propensities,
        initial_state=x0,
        t_max=T_MAX,
        n_trajectories=M_SSA,
    )
    demo_trajs.append(Trajectory.stack_on_grid(traj_list, time_grid))

max_species = max(crn.n_species for crn in demo_crns)
fig, axes = plt.subplots(max_species, len(demo_crns), figsize=(3 * len(demo_crns), 2.5 * max_species))
if max_species == 1:
    axes = axes[None, :]
if len(demo_crns) == 1:
    axes = axes[:, None]
t_np = time_grid.numpy()

for col, (crn, traj) in enumerate(zip(demo_crns, demo_trajs)):
    for s in range(crn.n_species):
        ax = axes[s, col]
        for m in range(M_SSA):
            ax.plot(t_np, traj[m, :, s].numpy(), alpha=0.4, lw=0.8, color="steelblue")
        ax.set_title(f"CRN {col} – {crn.species_names[s]}")
    for s in range(crn.n_species, max_species):
        axes[s, col].set_visible(False)

plt.tight_layout()
plt.show()

## Section 3: Train a small model

In [ ]:
# Build a tiny training dataset (30 train, 10 val)
def make_item(crn, x0, traj):
    return TrajectoryItem(
        crn_repr=crn_to_tensor_repr(crn),
        initial_state=x0,
        trajectories=traj,
        times=time_grid,
        motif_label="mass_action",
    )

all_crns = gen.sample_batch(40)
all_items = []
for crn in all_crns:
    x0 = gen.sample_initial_state(crn)
    traj_list = ssa.simulate_batch(
        stoichiometry=crn.stoichiometry_matrix,
        propensity_fn=crn.evaluate_propensities,
        initial_state=x0,
        t_max=T_MAX,
        n_trajectories=8,
    )
    traj = Trajectory.stack_on_grid(traj_list, time_grid)
    all_items.append(make_item(crn, x0, traj))

train_dataset = CRNTrajectoryDataset(all_items[:30])
val_dataset = CRNTrajectoryDataset(all_items[30:])
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [ ]:
# Tiny model: d_model=16
model_config = ModelConfig(
    encoder=EncoderConfig(d_model=16, n_layers=2, use_attention=False),
    sde=SDEConfig(d_model=16, d_hidden=32, n_noise_channels=MAX_N_REACTIONS, n_hidden_layers=1),
)
encoder = BipartiteGNNEncoder(model_config.encoder)
sde = CRNNeuralSDE(model_config.sde, n_species=MAX_N_SPECIES)

train_config = TrainingConfig(
    lr=1e-3,
    max_epochs=20,
    batch_size=8,
    n_sde_samples=4,
    dt=0.2,
    val_every=5,
    scheduler_type=SchedulerType.COSINE,
    use_wandb=False,
)
trainer = Trainer(encoder, sde, model_config, train_config)
result = trainer.train(train_dataset, val_dataset)

# Training curve
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(result.train_losses, label="train NLL")
if result.val_losses:
    ax.plot(result.val_epochs, result.val_losses, "o-", label="val rollout")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("Training curve")
ax.legend()
plt.tight_layout()
plt.show()

## Section 4: Evaluate on novel CRNs

In [ ]:
encoder.eval()
sde.eval()
solver = EulerMaruyamaSolver(model_config.sde)

novel_crns = gen.sample_batch(3)
K_ROLLOUTS = 10
M_NOVEL_SSA = 15

fig, axes = plt.subplots(len(novel_crns), MAX_N_SPECIES, figsize=(4 * MAX_N_SPECIES, 3 * len(novel_crns)))
if len(novel_crns) == 1:
    axes = axes[None, :]
t_np = time_grid.numpy()

for row, crn in enumerate(novel_crns):
    x0 = gen.sample_initial_state(crn)
    crn_repr = crn_to_tensor_repr(crn)

    with torch.no_grad():
        ctx = encoder(crn_repr, x0)
        padded_init = torch.zeros(MAX_N_SPECIES)
        padded_init[: crn.n_species] = x0
        sde_trajs = [
            solver.solve(sde, padded_init, ctx, time_grid, 0.2).states[:, : crn.n_species]
            for _ in range(K_ROLLOUTS)
        ]
    sde_tensor = torch.stack(sde_trajs)

    ssa_list = ssa.simulate_batch(
        stoichiometry=crn.stoichiometry_matrix,
        propensity_fn=crn.evaluate_propensities,
        initial_state=x0,
        t_max=T_MAX,
        n_trajectories=M_NOVEL_SSA,
    )
    ssa_tensor = Trajectory.stack_on_grid(ssa_list, time_grid)

    for s in range(crn.n_species):
        ax = axes[row, s]
        for m in range(M_NOVEL_SSA):
            ax.plot(t_np, ssa_tensor[m, :, s].numpy(), color="steelblue", alpha=0.2, lw=0.8)
        ax.plot(t_np, ssa_tensor[:, :, s].mean(0).numpy(), color="steelblue", lw=2, label="SSA")
        for k in range(K_ROLLOUTS):
            ax.plot(t_np, sde_tensor[k, :, s].numpy(), color="tomato", alpha=0.3, lw=0.8)
        ax.plot(t_np, sde_tensor[:, :, s].mean(0).numpy(), color="tomato", lw=2, label="SDE")
        ax.set_title(f"Novel CRN {row} – {crn.species_names[s]}")
        if s == 0:
            ax.legend(fontsize=7)
    for s in range(crn.n_species, MAX_N_SPECIES):
        axes[row, s].set_visible(False)

plt.tight_layout()
plt.show()

## Section 5: Summary

This notebook demonstrated the full CRN surrogate pipeline:

1. **Random CRN generation** — `MassActionCRNGenerator` produces chemically plausible CRNs with production, degradation, and participation constraints.
2. **SSA simulation** — `GillespieSSA` produces ground-truth stochastic trajectories for each CRN.
3. **Model training** — The bipartite GNN encoder + neural SDE trained jointly on multi-topology data via teacher-forcing NLL. States are padded to `max_n_species` so one model handles all topologies.
4. **Evaluation on novel topologies** — The trained model is applied to CRNs not seen during training. SSA mean and variance should match the SDE's predictions if training was sufficient.

**Next steps:**
- Scale to the full dataset (500 train, 100 val) using `experiments/scripts/generate_dataset.py`.
- Train for 200 epochs with `experiments/scripts/train.py`.
- Evaluate quantitatively in `experiments/analysis/evaluate_run.ipynb`.